# 3.3 分词带来的问题 / Problems caused by tokenization

字母、数字和多语言在分词后变成了什么样子，以及这怎样影响模型的表现。

What letters, digits and other languages look like after tokenization, and how this affects model behaviour.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 观察字母、数字和中文的分词 / Observe how letters, digits and Chinese are tokenized

用 tiktoken 写五个观察用的小函数，在 GPT-2 的编码和较新的 o200k 编码上对比。测试覆盖五个现象，即同一个词带不带空格切法不同、把字母分开写后每个字母成为一个 token、数字被切成不规则的片段且多一位就错位、中文的 token 数因分词器而异、GPT-2 中一个汉字可能被切到不同的 token 里。

Write five small inspection functions with tiktoken and compare the GPT-2 encoding with the newer o200k encoding. The tests cover five phenomena, namely that a word is cut differently with and without a space, that letters written apart each become a token, that numbers are cut into irregular pieces that shift with one more digit, that the token count of Chinese depends on the tokenizer, and that in GPT-2 one Chinese character may be split across tokens.

需要联网下载模型。/ Downloads a model.

In [ ]:
import tiktoken


def token_bytes(enc, text):
    """The raw bytes of every token of `text`: a list of bytes objects.

    Use enc.encode(text) and enc.decode_single_token_bytes(token_id).
    """
    # TODO 1
    raise NotImplementedError


def tokens_per_char(enc, text):
    """Average number of tokens per character of `text` (a float)."""
    # TODO 2
    raise NotImplementedError


def digit_groups(enc, number):
    """How a string of digits is cut up: the number of digits in each token, as a list of ints."""
    # TODO 3
    raise NotImplementedError


def spell_out(word):
    """'cat' -> ' c a t': a space before every letter."""
    # TODO 4
    raise NotImplementedError


def letters_are_tokens(enc, word):
    """True if, after spell_out, every letter of `word` is a token of its own."""
    # TODO 5
    raise NotImplementedError

In [ ]:
# 检查 / checks
import tiktoken

gpt2 = tiktoken.get_encoding("gpt2")              # GPT-2, vocabulary 50,257
o200k = tiktoken.get_encoding("o200k_base")       # a newer OpenAI encoding, vocabulary about 200,000

# 1. Words are not made of letters. The same word is cut differently with and without a space.
assert token_bytes(gpt2, "strawberry") == [b"st", b"raw", b"berry"]
assert token_bytes(gpt2, " strawberry") == [b" strawberry"]
assert b"".join(token_bytes(o200k, "Tokenization isn't obvious")) == b"Tokenization isn't obvious"

# 2. The model can only see letters if the text is written so that each letter is a token.
assert spell_out("cat") == " c a t"
assert letters_are_tokens(gpt2, "strawberry") and letters_are_tokens(o200k, "strawberry")
assert len(gpt2.encode(spell_out("strawberry"))) == 10

# 3. Digits are cut into irregular chunks, and one extra digit shifts the chunks.
assert digit_groups(gpt2, "1234567") == [3, 2, 2]
assert digit_groups(gpt2, "12345678") == [3, 2, 3]
assert digit_groups(o200k, "1234567") == [3, 3, 1], "this encoding cuts from the left in threes"
assert digit_groups(o200k, "12345678901") == [3, 3, 3, 2]

# 4. Languages cost different numbers of tokens, and it depends on the tokenizer.
zh, en = "北京是中国的首都", "Beijing is the capital of China"
assert tokens_per_char(gpt2, zh) > 1.5, "GPT-2 needs almost two tokens per Chinese character"
assert tokens_per_char(o200k, zh) < 1.0
assert tokens_per_char(gpt2, en) < 0.3
assert len(o200k.encode(zh)) < len(o200k.encode(en)), "with the newer encoding this sentence is cheaper in Chinese"

# 5. In GPT-2 a single Chinese character can be split across tokens: some tokens are not valid text on their own.
broken = 0
for b in token_bytes(gpt2, "今天天气很好"):
    try:
        b.decode("utf-8")
    except UnicodeDecodeError:
        broken += 1
assert broken >= 4
print("m33_observe: all tests passed")
print("  gpt2  1234567 ->", [b.decode() for b in token_bytes(gpt2, "1234567")])
print("  o200k 1234567 ->", [b.decode() for b in token_bytes(o200k, "1234567")])
print(f"  tokens per character, zh: gpt2 {tokens_per_char(gpt2, zh):.2f}, o200k {tokens_per_char(o200k, zh):.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`enc.decode_single_token_bytes(t)` 返回一个 token 的原始字节。这里用字节而不是字符串，是因为有些 token 单独解码不是合法的 UTF-8。

`enc.decode_single_token_bytes(t)` returns the raw bytes of one token. Bytes are used here and not strings because some tokens alone are not valid UTF-8.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`digit_groups` 就是每个 token 的字节长度。数字都是单字节字符，所以字节数等于位数。

`digit_groups` is the byte length of every token. Digits are single-byte characters, so the number of bytes equals the number of digits.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`letters_are_tokens` 比较的是 `spell_out(word)` 的 token 是否正好是每个字母前面加一个空格。

`letters_are_tokens` checks whether the tokens of `spell_out(word)` are exactly each letter with a space in front.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import tiktoken


def token_bytes(enc, text):
    return [enc.decode_single_token_bytes(t) for t in enc.encode(text)]


def tokens_per_char(enc, text):
    return len(enc.encode(text)) / len(text)


def digit_groups(enc, number):
    return [len(b) for b in token_bytes(enc, number)]


def spell_out(word):
    return "".join(" " + c for c in word)


def letters_are_tokens(enc, word):
    return [b.decode() for b in token_bytes(enc, spell_out(word))] == [" " + c for c in word]
```

</details>

## 2. 失败案例分析 / Analysing failure cases

根据上一题观察到的切分结果回答。
(a) 用户问 strawberry 里有几个字母 r。在 GPT-2 的分词下，这个词（不带前导空格）进入模型时是什么样子？模型的输入里有没有直接给出它的拼写？
(b) 给出两种让模型更容易答对这道题的办法，并说明各自为什么有效。
(c) 用 o200k 编码时，`1234567` 和 `12345678` 分别被切成什么？指出为什么这种切法对加法不利。
(d) 一位同学说“模型数不清字母，完全是分词造成的”。根据本模块的内容，这个说法需要怎样修正？

Answer from the tokenizations observed in the previous exercise.
(a) A user asks how many letters r are in strawberry. Under GPT-2's tokenization, what does this word (without a leading space) look like when it enters the model? Is its spelling given directly anywhere in the input?
(b) Give two ways of making it easier for the model to answer this question correctly, and say why each works.
(c) With the o200k encoding, how are `1234567` and `12345678` cut? Point out why this cutting is unfavourable for addition.
(d) A fellow student says that models cannot count letters purely because of tokenization. How should this statement be corrected in the light of this module?

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) 一种办法改变输入的写法，另一种办法把计数交给别的东西来做。

(b) One way changes how the input is written, the other hands the counting to something else.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 想一想个位数字落在哪个 token 里，以及它是否随数字长度变化。

(c) Consider which token holds the units digit and whether this changes with the length of the number.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 它是三个 token，即 st、raw、berry，进入模型的是这三个 token 的编号和对应的向量。输入中没有任何地方直接写出这些 token 由哪些字母组成，拼写只能靠模型从训练中间接学到。

(b) 其一，让模型先把单词逐个字母拼出来再计数，或者直接在提问时把字母用空格分开。这样每个字母是一个单独的 token，字母对模型可见，计数变成对可见的 token 逐个核对。其二，让模型调用代码执行工具，用一行程序来数。这把模型不擅长的精确计数交给了确定性的程序。

(c) 分别是 123、456、7 和 123、456、78。前两个 token 完全相同，但在两个数里代表的数量级不同，个位数字在一个数里单独成一个 token，在另一个数里与十位合在一起。加法需要按数位对齐并从右往左进位，这种从左往右的切分使对齐关系随数字长度变化。

(d) 分词是重要原因之一，它使字符层面的信息不直接可见。但已有研究的结论并不一致，有的研究发现错误与 token 数量的相关性较弱，而与单词长度和字母重复次数的相关性更强，模型能认出字母，困难在于计数。更准确的说法是分词与模型不擅长逐项计数这两个因素共同起作用。

---

(a) It is three tokens, st, raw and berry, and what enters the model are the ids of these three tokens and their vectors. Nowhere in the input is it written directly which letters these tokens consist of. The spelling has to be learned indirectly from training.

(b) First, have the model spell the word letter by letter before counting, or separate the letters with spaces in the question. Each letter is then a token of its own, the letters are visible to the model and counting becomes checking visible tokens one by one. Second, let the model call a code-execution tool and count with a line of code. This hands exact counting, which the model is not good at, to a deterministic program.

(c) They are 123, 456, 7 and 123, 456, 78. The first two tokens are identical yet stand for different orders of magnitude in the two numbers, and the units digit is a token of its own in one number and joined with the tens digit in the other. Addition needs alignment by place value and carries from right to left, and this left-to-right cutting makes the alignment change with the length of the number.

(d) Tokenization is one important cause, as it makes character-level information not directly visible. Yet existing studies do not agree. Some find that errors correlate only weakly with the number of tokens and more strongly with the length of the word and the repetition of letters, that models can recognize the letters and that the difficulty lies in counting. A more accurate statement is that tokenization and the weakness of models at item-by-item counting act together.

</details>